In [0]:
import requests
from pyspark.sql.avro.functions import from_avro
from pyspark.sql.functions import col, expr, current_timestamp, hour, to_timestamp
from delta.tables import DeltaTable

REDPANDA_BOOTSTRAP = "d8rtvni0pa8fqqs9v5n0.any.us-east-1.mpx.prd.cloud.redpanda.com:9092"
SCHEMA_REGISTRY_URL = "https://d8rtvni0pa8fqqs9v5n0.registry.us-east-1.mpx.prd.cloud.redpanda.com"
REDPANDA_USERNAME = "ecommerce-producer"
REDPANDA_PASSWORD = dbutils.secrets.get(scope="redpanda", key="password")

BRONZE_USER_ACTIVITY_PATH = "/Volumes/workspace/default/ecommerce_data/bronze/user_activity"
SILVER_USER_ACTIVITY_PATH = "/Volumes/workspace/default/ecommerce_data/silver/user_activity"
CHECKPOINT_SILVER_USER_ACTIVITY = "/Volumes/workspace/default/ecommerce_data/checkpoints/silver_user_activity"

VALID_EVENT_TYPES = ["product_view", "add_to_cart", "checkout", "payment", "search", "login", "review"]

In [0]:
bronze_stream = spark.readStream \
    .format("delta") \
    .load(BRONZE_USER_ACTIVITY_PATH)

In [0]:
def clean_batch(batch_df, batch_id):
    if batch_df.isEmpty():
        return

    cleaned = batch_df \
        .filter(col("event_id").isNotNull()) \
        .filter(col("user_id").isNotNull()) \
        .filter(col("event_type").isin(VALID_EVENT_TYPES)) \
        .withColumn("event_hour", hour(to_timestamp(col("timestamp_ms") / 1000))) \
        .dropDuplicates(["event_id"])

    if DeltaTable.isDeltaTable(spark, SILVER_USER_ACTIVITY_PATH):
        silver_table = DeltaTable.forPath(spark, SILVER_USER_ACTIVITY_PATH)
        silver_table.alias("s").merge(
            cleaned.alias("n"),
            "s.event_id = n.event_id"
        ).whenNotMatchedInsertAll().execute()
    else:
        cleaned.write.format("delta").save(SILVER_USER_ACTIVITY_PATH)

query_silver = bronze_stream.writeStream \
    .foreachBatch(clean_batch) \
    .option("checkpointLocation", CHECKPOINT_SILVER_USER_ACTIVITY) \
    .trigger(availableNow=True) \
    .start()

query_silver.awaitTermination()
print("Silver processing complete")

In [0]:
silver_df = spark.read.format("delta").load(SILVER_USER_ACTIVITY_PATH)
print(f"Total rows in Silver user_activity: {silver_df.count()}")
silver_df.show(10, truncate=False)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/ecommerce_data/"))